# 🧪 [Day 27] 엔터프라이즈 지식 그래프 실전 핸즈온 워크북

> **실습 시나리오**: 당신은 글로벌 K-컬처 엔터테인먼트 사의 수석 데이터 엔지니어입니다.  
> 기존 RDB의 한계를 극복하고, **LPG 모델링부터 2홉 실시간 추천, Pydantic 온톨로지 품질 거버넌스, RDF 시맨틱 트리플, SPARQL 6대 쿼리, 그리고 GraphRAG 사실 기반 AI 시스템**까지 엔드투엔드 파이프라인을 직접 구축합니다.

---

## 🛠️ [미션 0] 필수 라이브러리 로드 및 환경 준비
아래 셀을 실행하여 실습에 필요한 `pydantic`, `rdflib`, `networkx`, `matplotlib`을 준비합니다.

In [ ]:
import sys
from pathlib import Path
from typing import Literal

from pydantic import BaseModel, Field, ValidationError
import rdflib
from rdflib import Graph, URIRef, Literal as RDFLiteral, Namespace
from rdflib.namespace import RDF, RDFS
import networkx as nx
import matplotlib.pyplot as plt

print("✅ 환경 준비 완료!")

---
## 🧱 [미션 1] 속성 그래프 모델링 (LPG: Labeled Property Graph)

### 🎯 미션 목표
- 사용자 3명(`u1~u3`), 곡 5개(`s1~s5`), 아티스트 3명(`a1~a3`)의 정보를 담는 **`nodes` 딕셔너리**를 정의하세요.
- 사용자의 청취(`들었다`)와 곡의 가수(`부른가수`) 관계를 나타내는 **`edges` 튜플 리스트**를 정의하세요.

### 💡 가이드
- `nodes`는 `{ id: {'label': 종류, 'props': {'이름'/'제목': 값, ...}} }` 형태입니다.
- `edges`는 `[ (출발ID, 관계명, 도착ID), ... ]` 튜플 리스트 형태입니다.

In [ ]:
# [TODO] 미션 1: nodes 딕셔너리와 edges 리스트를 작성하세요.
nodes = {
    # 사용자 (u1: 민서/서울, u2: 준우/부산, u3: 하은/서울)
    'u1': {'label': '사용자', 'props': {'이름': '민서', '지역': '서울'}},
    'u2': {'label': '사용자', 'props': {'이름': '준우', '지역': '부산'}},
    'u3': {'label': '사용자', 'props': {'이름': '하은', '지역': '서울'}},
    
    # 곡 (s1: 밤편지, s2: 좋은 날, s3: Ditto, s4: OMG, s5: Dynamite)
    's1': {'label': '곡', 'props': {'제목': '밤편지', '장르': '발라드'}},
    's2': {'label': '곡', 'props': {'제목': '좋은 날', '장르': '댄스'}},
    's3': {'label': '곡', 'props': {'제목': 'Ditto', '장르': 'K-POP'}},
    's4': {'label': '곡', 'props': {'제목': 'OMG', '장르': 'K-POP'}},
    's5': {'label': '곡', 'props': {'제목': 'Dynamite', '장르': '디스코'}},
    
    # 아티스트 (a1: 아이유, a2: 뉴진스, a3: BTS)
    'a1': {'label': '아티스트', 'props': {'이름': '아이유', '국적': '대한민국'}},
    'a2': {'label': '아티스트', 'props': {'이름': '뉴진스', '국적': '대한민국'}},
    'a3': {'label': '아티스트', 'props': {'이름': 'BTS', '국적': '대한민국'}},
}

edges = [
    ('u1', '들었다', 's1'),
    ('u1', '들었다', 's3'),
    ('u2', '들었다', 's3'),
    ('u2', '들었다', 's4'),
    ('u2', '들었다', 's5'),
    ('u3', '들었다', 's1'),
    ('s1', '부른가수', 'a1'),
    ('s2', '부른가수', 'a1'),
    ('s3', '부른가수', 'a2'),
    ('s4', '부른가수', 'a2'),
    ('s5', '부른가수', 'a3'),
]

print(f"노드 수: {len(nodes)}, 엣지 수: {len(edges)}")

In [ ]:
# [자가채점] 미션 1
assert len(nodes) == 11, 'nodes에는 총 11개의 노드가 정의되어야 합니다.'
assert len(edges) == 11, 'edges에는 총 11개의 관계가 정의되어야 합니다.'
assert nodes['u1']['props']['이름'] == '민서', 'u1의 이름은 민서여야 합니다.'
assert ('u1', '들었다', 's3') in edges, 'u1이 s3를 들은 관계가 포함되어야 합니다.'
print('✅ [미션 1 통과!] LPG 모델링이 완벽히 구축되었습니다.')

---
## ⚡ [미션 2] 고속 그래프 순회 & 실시간 2홉 음악 추천 엔진 구현

### 🎯 미션 목표
1. `edges`로부터 `들었다` 관계에 대한 **정방향 인접 사전(`adjacency`: 사용자 -> 곡 목록)**과 **역방향 인접 사전(`reverse_adj`: 곡 -> 청취자 목록)**을 만드세요.
2. 특정 사용자(예: `u1` 민서)에게 **"내가 들은 곡을 들은 다른 사용자들이 청취한 곡 중, 내가 안 들은 곡"**을 추천하는 2홉 추천 로직을 완성하세요.

In [ ]:
# [TODO] 미션 2: 인접 사전 구축 및 2홉 추천 로직 작성
adjacency = {}       # 사용자 -> 들은 곡 목록
reverse_adj = {}     # 곡 -> 들은 사용자 목록

for src, rel, dst in edges:
    if rel == '들었다':
        adjacency.setdefault(src, []).append(dst)
        reverse_adj.setdefault(dst, []).append(src)

def get_2hop_recommendations(target_user: str) -> list[str]:
    my_songs = set(adjacency.get(target_user, []))
    recommended = set()
    
    # 1홉 (내가 들은 곡)
    for s in my_songs:
        # 역방향 1홉 (그 곡을 들은 다른 사람)
        for other in reverse_adj.get(s, []):
            if other == target_user:
                continue
            # 2홉 (그 사람이 들은 다른 곡)
            for candidate in adjacency.get(other, []):
                if candidate not in my_songs:
                    recommended.add(candidate)
                    
    return sorted([nodes[sid]['props']['제목'] for sid in recommended])

u1_rec = get_2hop_recommendations('u1')
print('민서(u1) 2홉 추천 곡 목록:', u1_rec)

In [ ]:
# [자가채점] 미션 2
assert len(adjacency['u1']) == 2, 'u1은 2곡(s1, s3)을 들었어야 합니다.'
assert len(reverse_adj['s3']) == 2, 's3(Ditto)는 u1과 u2가 들었어야 합니다.'
assert set(u1_rec) == {'OMG', 'Dynamite'}, 'u1 추천 곡은 OMG, Dynamite 2곡이어야 합니다.'
print('✅ [미션 2 통과!] JOIN 없는 $O(1)$ 고속 2홉 추천 엔진이 완벽히 동작합니다.')

---
## 📐 [미션 3] Pydantic 기반 온톨로지 스키마 거버넌스

### 🎯 미션 목표
- 외부 수집 원천 사실(`raw_facts`)에 대해 `Pydantic` 스키마 모델을 정의하고, **불량 데이터('외계인', '무국적')를 걸러내어 `clean_facts` 리스트를 생성**하세요.

In [ ]:
# [TODO] 미션 3: Pydantic 온톨로지 검증 클래스 작성 및 필터링
ALLOWED_PREDICATES = {'들었다', '부른가수', '직업', '국적'}
ALLOWED_JOBS = {'가수', '배우', '영화감독'}

class OntologyTriple(BaseModel):
    subject: str
    predicate: str
    object_value: str

    def is_valid(self) -> bool:
        if self.predicate not in ALLOWED_PREDICATES:
            return False
        if self.predicate == '직업' and self.object_value not in ALLOWED_JOBS:
            return False
        if self.predicate == '국적' and self.object_value in {'무국적', '알수없음'}:
            return False
        return True

raw_facts = [
    {'subject': '민서', 'predicate': '직업', 'object_value': '가수'},
    {'subject': '민서', 'predicate': '국적', 'object_value': '대한민국'},
    {'subject': '하은', 'predicate': '직업', 'object_value': '배우'},
    {'subject': '하은', 'predicate': '국적', 'object_value': '대한민국'},
    {'subject': '준우', 'predicate': '직업', 'object_value': '가수'},
    {'subject': '준우', 'predicate': '국적', 'object_value': '미국'},
    {'subject': '아이유', 'predicate': '직업', 'object_value': '가수'},
    {'subject': '아이유', 'predicate': '국적', 'object_value': '대한민국'},
    {'subject': '뉴진스', 'predicate': '직업', 'object_value': '가수'},
    {'subject': '뉴진스', 'predicate': '국적', 'object_value': '대한민국'},
    {'subject': 'BTS', 'predicate': '직업', 'object_value': '가수'},
    {'subject': 'BTS', 'predicate': '국적', 'object_value': '대한민국'},
    {'subject': '괴도루팡', 'predicate': '직업', 'object_value': '외계인'},  # ❌
    {'subject': '유령회원', 'predicate': '국적', 'object_value': '무국적'},  # ❌
]

clean_facts = []
for item in raw_facts:
    try:
        t = OntologyTriple(**item)
        if t.is_valid():
            clean_facts.append(t)
    except ValidationError:
        pass

print(f"원본 사실: {len(raw_facts)}건 -> 온톨로지 통과 사실: {len(clean_facts)}건")

In [ ]:
# [자가채점] 미션 3
assert len(clean_facts) == 12, '유효 사실은 불량 2건이 제거된 12건이어야 합니다.'
assert all(f.object_value not in {'외계인', '무국적'} for f in clean_facts), '불량 데이터가 남아있습니다.'
print('✅ [미션 3 통과!] 온톨로지 품질 거버넌스가 불량 데이터를 완벽히 차단했습니다.')

---
## 💎 [미션 4] 국제 표준 RDF 시맨틱 트리플 그래프 구축

### 🎯 미션 목표
- `rdflib.Graph`를 생성하고, `clean_facts`와 음악 청취 관계(`nodes`, `edges`)를 **RDF 시맨틱 트리플 (주어, 술어, 목적어)**로 주입하세요.

In [ ]:
# [TODO] 미션 4: rdflib Graph 생성 및 트리플 적재
kg = Graph()
EX = Namespace("http://example.org/")

# 1. 온톨로지 통과 사실 적재
for f in clean_facts:
    s = EX[f.subject]
    p = EX[f.predicate]
    o = RDFLiteral(f.object_value) if f.predicate == '국적' else EX[f.object_value]
    kg.add((s, p, o))

# 2. 사용자 타입 및 음악 청취/곡 정보 적재
kg.add((EX.민서, RDF.type, EX.사용자))
kg.add((EX.준우, RDF.type, EX.사용자))
kg.add((EX.하은, RDF.type, EX.사용자))

kg.add((EX.민서, EX.들었다, EX.s1))
kg.add((EX.민서, EX.들었다, EX.s3))
kg.add((EX.준우, EX.들었다, EX.s3))
kg.add((EX.준우, EX.들었다, EX.s4))
kg.add((EX.준우, EX.들었다, EX.s5))

kg.add((EX.s1, EX.제목, RDFLiteral("밤편지")))
kg.add((EX.s2, EX.제목, RDFLiteral("좋은 날")))
kg.add((EX.s3, EX.제목, RDFLiteral("Ditto")))
kg.add((EX.s4, EX.제목, RDFLiteral("OMG")))
kg.add((EX.s5, EX.제목, RDFLiteral("Dynamite")))

kg.add((EX.s1, EX.부른가수, EX.아이유))
kg.add((EX.s2, EX.부른가수, EX.아이유))
kg.add((EX.s3, EX.부른가수, EX.뉴진스))
kg.add((EX.s4, EX.부른가수, EX.뉴진스))
kg.add((EX.s5, EX.부른가수, EX.BTS))

# OPTIONAL 실습용 속성: 민서만 인스타그램 보유
kg.add((EX.민서, EX.인스타그램, RDFLiteral("@minseo_official")))

print(f"적재된 RDF 트리플 총 개수: {len(kg)}")

In [ ]:
# [자가채점] 미션 4
assert len(kg) == 31, f'트리플 총 개수는 31개여야 합니다. 현재: {len(kg)}개'
print('✅ [미션 4 통과!] 국제 표준 RDF 시맨틱 트리플 그래프가 성공적으로 적재되었습니다.')

---
## 🔍 [미션 5] 표준 SPARQL 1.1 질의 6대 실무 패턴 정복

### 🎯 미션 목표
아래 6개의 SPARQL 질의를 작성하고 실행 결과를 자가채점으로 확인하세요.
1. **기본 조인**: 대한민국 국적의 가수 찾기 (`q1`)
2. **`UNION`**: 미국 국적이거나 직업이 배우인 인물 (`q2`)
3. **`OPTIONAL`**: 사용자와 인스타그램 계정 (`q3`)
4. **`FILTER NOT EXISTS`**: Ditto(s3)를 안 들은 사용자 (`q4`)
5. **속성 경로 (`/`)**: 민서가 들은 노래의 가수 목록 (`q5`)
6. **집계 (`GROUP BY` & `COUNT`)**: 아티스트별 총 청취수 (`q6`)

In [ ]:
# [TODO] 미션 5-1: 기본 조인 (대한민국 국적의 가수)
q1 = """
PREFIX ex: <http://example.org/>
SELECT ?name WHERE {
    ?p ex:직업 ex:가수 .
    ?p ex:국적 "대한민국" .
    BIND(STRAFTER(STR(?p), "http://example.org/") AS ?name)
}
"""
res1 = [str(r.name) for r in kg.query(q1)]
print("1. 대한민국 국적 가수:", res1)

In [ ]:
# [TODO] 미션 5-2: UNION (미국 국적이거나 배우)
q2 = """
PREFIX ex: <http://example.org/>
SELECT DISTINCT ?name WHERE {
    { ?p ex:국적 "미국" }
    UNION
    { ?p ex:직업 ex:배우 }
    BIND(STRAFTER(STR(?p), "http://example.org/") AS ?name)
}
"""
res2 = [str(r.name) for r in kg.query(q2)]
print("2. 미국 국적이거나 배우:", res2)

In [ ]:
# [TODO] 미션 5-3: OPTIONAL (인스타그램)
q3 = """
PREFIX ex: <http://example.org/>
SELECT ?name ?insta WHERE {
    ?u a ex:사용자 .
    BIND(STRAFTER(STR(?u), "http://example.org/") AS ?name)
    OPTIONAL { ?u ex:인스타그램 ?insta }
}
"""
res3 = {str(r.name): (str(r.insta) if r.insta else None) for r in kg.query(q3)}
print("3. 사용자 인스타그램:", res3)

In [ ]:
# [TODO] 미션 5-4: FILTER NOT EXISTS (Ditto 안 들은 사용자)
q4 = """
PREFIX ex: <http://example.org/>
SELECT ?name WHERE {
    ?u a ex:사용자 .
    FILTER NOT EXISTS { ?u ex:들었다 ex:s3 }
    BIND(STRAFTER(STR(?u), "http://example.org/") AS ?name)
}
"""
res4 = [str(r.name) for r in kg.query(q4)]
print("4. Ditto 안 들은 사용자:", res4)

In [ ]:
# [TODO] 미션 5-5: 속성 경로 / (민서가 들은 노래의 가수)
q5 = """
PREFIX ex: <http://example.org/>
SELECT DISTINCT ?artist WHERE {
    ex:민서 ex:들었다/ex:부른가수 ?artistNode .
    BIND(STRAFTER(STR(?artistNode), "http://example.org/") AS ?artist)
}
"""
res5 = [str(r.artist) for r in kg.query(q5)]
print("5. 민서가 들은 노래의 가수 (속성 경로):", res5)

In [ ]:
# [TODO] 미션 5-6: COUNT & GROUP BY (아티스트별 총 청취수)
q6 = """
PREFIX ex: <http://example.org/>
SELECT ?artist (COUNT(?u) AS ?cnt) WHERE {
    ?u ex:들었다/ex:부른가수 ?artistNode .
    BIND(STRAFTER(STR(?artistNode), "http://example.org/") AS ?artist)
}
GROUP BY ?artist
ORDER BY DESC(?cnt)
"""
res6 = {str(r.artist): int(r.cnt) for r in kg.query(q6)}
print("6. 아티스트별 총 청취수:", res6)

In [ ]:
# [자가채점] 미션 5 (6대 쿼리 통합 검증)
assert set(res1) == {'민서', '아이유', '뉴진스', 'BTS'}, 'q1 결과가 올바르지 않습니다.'
assert set(res2) == {'준우', '하은'}, 'q2 결과가 올바르지 않습니다.'
assert res3['민서'] == '@minseo_official' and res3['준우'] is None, 'q3 OPTIONAL 결과가 올바르지 않습니다.'
assert res4 == ['하은'], 'q4 Ditto를 안 들은 사람은 하은이어야 합니다.'
assert set(res5) == {'아이유', '뉴진스'}, 'q5 민서가 들은 가수는 아이유, 뉴진스여야 합니다.'
assert res6['뉴진스'] == 3 and res6['아이유'] == 1 and res6['BTS'] == 1, 'q6 집계 결과가 올바르지 않습니다.'
print('✅ [미션 5 통과!] SPARQL 6대 실무 질의 패턴을 완벽히 마스터했습니다.')

---
## 🤖 [미션 6] 엔터프라이즈 GraphRAG 사실 기반 프롬프트 엔진 연동

### 🎯 미션 목표
- 사용자 이름을 받아 **지식 그래프에서 해당 사용자의 청취 곡, 가수, 가수의 국적 사실 서브그래프를 SPARQL로 추출하고, 환각 0%를 보장하는 프롬프트를 조립하는 `execute_graph_rag` 함수**를 작성하세요.

In [ ]:
# [TODO] 미션 6: GraphRAG 엔진 함수 구현
def execute_graph_rag(user_name: str, question: str) -> tuple[str, str]:
    query = f"""
    PREFIX ex: <http://example.org/>
    SELECT ?songTitle ?artist ?country WHERE {{
        ex:{user_name} ex:들었다 ?song .
        ?song ex:제목 ?songTitle .
        ?song ex:부른가수 ?artistNode .
        ?artistNode ex:국적 ?country .
        BIND(STRAFTER(STR(?artistNode), "http://example.org/") AS ?artist)
    }}
    """
    rows = list(kg.query(query))
    facts = [f"• {user_name} -> 곡 '{r.songTitle}' -> 가수 {r.artist} (국적: {r.country})" for r in rows]
    context = "\n".join(facts)
    
    prompt = f"""
[지식 그래프 검증 팩트]:
{context}

[질문]: {question}
위 지식 그래프 사실에만 100% 근거하여 답변하라.
"""
    return context, prompt

context, prompt = execute_graph_rag('민서', '민서가 들은 노래와 가수의 국적을 알려줘.')
print("추출된 팩트:\n", context)

In [ ]:
# [자가채점] 미션 6
assert '아이유' in context and '뉴진스' in context, '민서 관련 팩트에 아이유와 뉴진스가 포함되어야 합니다.'
assert '대한민국' in context, '국적 정보가 포함되어야 합니다.'
assert '[지식 그래프 검증 팩트]' in prompt, '프롬프트 규격이 올바르게 생성되어야 합니다.'
print('✅ [미션 6 통과!] 환각 없는 GraphRAG 시스템이 성공적으로 구축되었습니다.')
print('\n🎉 축하합니다! Day 27 엔터프라이즈 지식 그래프 실전 핸즈온을 완벽히 완주하셨습니다!')